> Check individual signals according to the community they belong.

In [1]:
import pathlib

import yaml
import pandas as pd

import trapallada.util

So that `import`ed Python modules are automatically reloaded when modified

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


So that *plotly*'s pictures are correctly rendered

In [3]:
import plotly.io as pio
pio.renderers.default = 'notebook'

# Parameters

Directory with all the files

In [4]:
labels_dir = trapallada.util.get_base_directory() / 'labels'
assert labels_dir.exists()

The "selected by" file indicating the algorithm (VAE or AR) that selected every window.

In [5]:
selected_by_file = labels_dir / 'signals_to_assess.parquet'
assert selected_by_file.exists()

The file with the annotations provided by Javi López Santiago.

In [6]:
labels_file = labels_dir / 'res.xlsx'
assert labels_file.exists()

Output file *merging* the above files

In [7]:
output_file = labels_dir / 'annotated_windows.parquet'

# Input data

The *selected by* `DataFrame` stating the algorithm (VAE or AR) that selected every window.

In [8]:
selected_by_df = pd.read_parquet(selected_by_file)
selected_by_df

,start,end,tar file,source,signal,Selected by
14796,2004-02-29 23:34:36.224518,2004-03-01 05:17:59.744518,0201360201.tar,P0201360201PNX000SRCTSR8002.FTZ,0201360201/P0201360201PNX000SRCTSR8002,VAE
25125,2015-05-25 05:07:30.919731,2015-05-25 10:50:54.439731,0770180201.tar,P0770180201PNX000SRCTSR8001.FTZ,0770180201/P0770180201PNX000SRCTSR8001,AR
15937,2011-01-03 19:00:05.711776,2011-01-04 00:43:29.231776,0652330101.tar,P0652330101PNX000SRCTSR8001.FTZ,0652330101/P0652330101PNX000SRCTSR8001,VAE
22425,2015-03-26 05:54:06.667239,2015-03-26 11:37:30.187239,0764800301.tar,P0764800301PNX000SRCTSR8005.FTZ,0764800301/P0764800301PNX000SRCTSR8005,NaN
29039,2015-06-27 04:31:17.762896,2015-06-27 10:14:41.282896,0761970101.tar,P0761970101PNX000SRCTSR8002.FTZ,0761970101/P0761970101PNX000SRCTSR8002,NaN
...,...,...,...,...,...,...
27106,2014-10-04 23:21:51.077119,2014-10-05 05:05:14.597119,0728990101.tar,P0728990101PNX000SRCTSR8001.FTZ,0728990101/P0728990101PNX000SRCTSR8001,AR
9253,2008-06-27 08:47:46.841629,2008-06-27 14:31:10.361629,0551750301.tar,P0551750301PNX000SRCTSR8001.FTZ,0551750301/P0551750301PNX000SRCTSR8001,AR
15049,2017-10-24 15:17:34.836966,2017-10-24 21:00:58.356966,0795730101.tar,P0795730101PNX000SRCTSR8001.FTZ,0795730101/P0795730101PNX000SRCTSR8001,VAE
5371,2013-09-27 21:06:30.736658,2013-09-28 02:49:54.256658,0724380101.tar,P0724380101PNX000SRCTSR8003.FTZ,0724380101/P0724380101PNX000SRCTSR8003,NaN


The `DataFrame` with the labels provided by the annotator. Its index (window *id*) must be the same as that of the above `DataFrame`.

In [9]:
labels_df = pd.read_excel(labels_file)
labels_df = labels_df.drop(columns=labels_df.columns[0])
labels_df.index = selected_by_df.index
labels_df

,Label,Type,Subtype
14796,Interesting,doubleExponential,flare
25125,NaN,NaN,NaN
15937,Interesting,doubleExponential,flare
22425,Interesting,background,NaN
29039,NOT interesting,NaN,NaN
...,...,...,...
27106,NOT interesting,NaN,NaN
9253,NOT interesting,NaN,NaN
15049,Interesting,background,NaN
5371,NOT interesting,NaN,NaN


They are merged together

In [10]:
res_df = pd.concat((selected_by_df, labels_df), axis=1)
res_df

,start,end,tar file,source,signal,Selected by,Label,Type,Subtype
14796,2004-02-29 23:34:36.224518,2004-03-01 05:17:59.744518,0201360201.tar,P0201360201PNX000SRCTSR8002.FTZ,0201360201/P0201360201PNX000SRCTSR8002,VAE,Interesting,doubleExponential,flare
25125,2015-05-25 05:07:30.919731,2015-05-25 10:50:54.439731,0770180201.tar,P0770180201PNX000SRCTSR8001.FTZ,0770180201/P0770180201PNX000SRCTSR8001,AR,NaN,NaN,NaN
15937,2011-01-03 19:00:05.711776,2011-01-04 00:43:29.231776,0652330101.tar,P0652330101PNX000SRCTSR8001.FTZ,0652330101/P0652330101PNX000SRCTSR8001,VAE,Interesting,doubleExponential,flare
22425,2015-03-26 05:54:06.667239,2015-03-26 11:37:30.187239,0764800301.tar,P0764800301PNX000SRCTSR8005.FTZ,0764800301/P0764800301PNX000SRCTSR8005,NaN,Interesting,background,NaN
29039,2015-06-27 04:31:17.762896,2015-06-27 10:14:41.282896,0761970101.tar,P0761970101PNX000SRCTSR8002.FTZ,0761970101/P0761970101PNX000SRCTSR8002,NaN,NOT interesting,NaN,NaN
...,...,...,...,...,...,...,...,...,...
27106,2014-10-04 23:21:51.077119,2014-10-05 05:05:14.597119,0728990101.tar,P0728990101PNX000SRCTSR8001.FTZ,0728990101/P0728990101PNX000SRCTSR8001,AR,NOT interesting,NaN,NaN
9253,2008-06-27 08:47:46.841629,2008-06-27 14:31:10.361629,0551750301.tar,P0551750301PNX000SRCTSR8001.FTZ,0551750301/P0551750301PNX000SRCTSR8001,AR,NOT interesting,NaN,NaN
15049,2017-10-24 15:17:34.836966,2017-10-24 21:00:58.356966,0795730101.tar,P0795730101PNX000SRCTSR8001.FTZ,0795730101/P0795730101PNX000SRCTSR8001,VAE,Interesting,background,NaN
5371,2013-09-27 21:06:30.736658,2013-09-28 02:49:54.256658,0724380101.tar,P0724380101PNX000SRCTSR8003.FTZ,0724380101/P0724380101PNX000SRCTSR8003,NaN,NOT interesting,NaN,NaN


It is saved

In [11]:
res_df.to_parquet(output_file, index=True)

# Precision

The *sub*`DataFrame`s for VAE...

In [20]:
vae_selected_df = res_df[res_df['Selected by'] == 'VAE'].drop(columns=['Selected by'])

, AR...

In [21]:
ar_selected_df = res_df[res_df['Selected by'] == 'AR'].drop(columns=['Selected by'])

...and *random*

In [47]:
random_selected_df = res_df[pd.isna(res_df['Selected by'])].drop(columns=['Selected by'])
random_selected_df.shape

(82, 8)

Another *sub*`DataFrame` with those that were annotated as `Interesting`

In [37]:
interesting_df = res_df[res_df['Label'] == 'Interesting']
interesting_df.shape

(125, 9)

# Precision

## VAE

In [49]:
vae_precision = (vae_selected_df['Label'] == 'Interesting').sum().item() / len(vae_selected_df)
vae_precision

0.8833333333333333

## AR

In [50]:
ar_precision = (ar_selected_df['Label'] == 'Interesting').sum().item() / len(ar_selected_df)
ar_precision

0.46551724137931033

## Random

In [51]:
random_precision = (random_selected_df['Label'] == 'Interesting').sum().item() / len(random_selected_df)
random_precision

0.5487804878048781

# Recall

It requires a probability/score for every point...

In [52]:
# vae_recall = interesting_df.index.isin(vae_selected_df.index).sum().item()
# vae_recall